# GAN (DCGAN-style) on MNIST —

Standard PyTorch + torchvision + matplotlib + tqdm.


In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.manual_seed(0)
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
device


## Data

In [ ]:
batch_size = 128
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,)),
])
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
x, _ = next(iter(train_loader))
x.shape, x.min().item(), x.max().item()


In [ ]:
def show_batch(x, n=64, title=""):
    x = x[:n].detach().cpu()
    grid = make_grid(x, nrow=int(math.sqrt(n)), normalize=True, value_range=(-1,1))
    plt.figure(figsize=(6,6))
    plt.title(title)
    plt.axis("off")
    plt.imshow(grid.permute(1,2,0).squeeze(), cmap="gray")
    plt.show()

show_batch(x, title="MNIST (normalized to [-1,1])")


## Models

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=128, img_channels=1, base=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, base*4, 7, 1, 0, bias=False),
            nn.BatchNorm2d(base*4),
            nn.ReLU(True),
            nn.ConvTranspose2d(base*4, base*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base*2),
            nn.ReLU(True),
            nn.ConvTranspose2d(base*2, base, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(True),
            nn.Conv2d(base, img_channels, 3, 1, 1),
            nn.Tanh(),
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, img_channels=1, base=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, base, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base, base*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base*2, 1, 7, 1, 0),
        )
    def forward(self, x): return self.net(x).view(-1)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    if isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.constant_(m.bias, 0)

z_dim = 128
G = Generator(z_dim=z_dim).to(device).apply(init_weights)
D = Discriminator().to(device).apply(init_weights)
(sum(p.numel() for p in G.parameters()), sum(p.numel() for p in D.parameters()))


## Train

In [ ]:
bce = nn.BCEWithLogitsLoss()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

@dataclass
class TrainConfig:
    epochs: int = 3
    log_every: int = 200
cfg = TrainConfig()

fixed_z = torch.randn(64, z_dim, 1, 1, device=device)

G.train(); D.train()
step = 0
for epoch in range(cfg.epochs):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}")
    for x_real, _ in pbar:
        x_real = x_real.to(device)
        bs = x_real.size(0)

        # D
        z = torch.randn(bs, z_dim, 1, 1, device=device)
        x_fake = G(z).detach()
        loss_D = bce(D(x_real), torch.ones(bs, device=device)) + bce(D(x_fake), torch.zeros(bs, device=device))
        opt_D.zero_grad(set_to_none=True); loss_D.backward(); opt_D.step()

        # G
        z = torch.randn(bs, z_dim, 1, 1, device=device)
        x_fake = G(z)
        loss_G = bce(D(x_fake), torch.ones(bs, device=device))
        opt_G.zero_grad(set_to_none=True); loss_G.backward(); opt_G.step()

        if step % cfg.log_every == 0:
            pbar.set_postfix({"loss_D": f"{loss_D.item():.3f}", "loss_G": f"{loss_G.item():.3f}"})
        step += 1

    G.eval()
    with torch.no_grad():
        x_vis = G(fixed_z).cpu()
    show_batch(x_vis, title=f"Samples after epoch {epoch+1}")
    G.train()
